<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/Monica/MovieRecommendationCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
import matplotlib.pyplot as plt

# Download and extract the dataset
movielens_data_file_url = "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
movielens_zipped_file = keras.utils.get_file("ml-latest-small.zip", movielens_data_file_url, extract=False)

keras_datasets_path = Path(movielens_zipped_file).parents[0]
movielens_dir = keras_datasets_path / "ml-latest-small"

if not movielens_dir.exists():
    with ZipFile(movielens_zipped_file, "r") as zip:
        print("Extracting all the files now...")
        zip.extractall(path=keras_datasets_path)
        print("Done!")

# Load data into DataFrames
ratings_file = movielens_dir / "ratings.csv"
df = pd.read_csv(ratings_file)
movie_df = pd.read_csv(movielens_dir / "movies.csv")

978202/978202 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Extracting all the files now...
Done!


In [ ]:
# Map user and movie IDs to indices
user_ids = df["userId"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
movie_ids = df["movieId"].unique().tolist()
movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}
movie_encoded2movie = {i: x for i, x in enumerate(movie_ids)}

df["user"] = df["userId"].map(user2user_encoded)
df["movie"] = df["movieId"].map(movie2movie_encoded)

num_users = len(user2user_encoded)
num_movies = len(movie_encoded2movie)
df["rating"] = df["rating"].values.astype(np.float32)

# Normalize ratings
min_rating = min(df["rating"])
max_rating = max(df["rating"])
df = df.sample(frac=1, random_state=42)
x = df[["user", "movie"]].values
y = df["rating"].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values

# Split into training and validation sets
train_indices = int(0.9 * df.shape[0])
x_train, x_val, y_train, y_val = (x[:train_indices], x[train_indices:], y[:train_indices], y[train_indices:])

In [ ]:
EMBEDDING_SIZE = 50

class RecommenderNet(keras.Model):
    def __init__(self, num_users, num_movies, embedding_size, **kwargs):
        super(RecommenderNet, self).__init__(**kwargs)
        self.user_embedding = layers.Embedding(
            num_users, embedding_size, embeddings_initializer="he_normal",
            embeddings_regularizer=keras.regularizers.l2(1e-6)
        )
        self.user_bias = layers.Embedding(num_users, 1)
        self.movie_embedding = layers.Embedding(
            num_movies, embedding_size, embeddings_initializer="he_normal",
            embeddings_regularizer=keras.regularizers.l2(1e-6)
        )
        self.movie_bias = layers.Embedding(num_movies, 1)

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        user_bias = self.user_bias(inputs[:, 0])
        movie_vector = self.movie_embedding(inputs[:, 1])
        movie_bias = self.movie_bias(inputs[:, 1])
        dot_user_movie = tf.tensordot(user_vector, movie_vector, 2)
        x = dot_user_movie + user_bias + movie_bias
        return tf.nn.sigmoid(x)

model = RecommenderNet(num_users, num_movies, EMBEDDING_SIZE)
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=0.001)
)

In [ ]:
history = model.fit(
    x=x_train,
    y=y_train,
    batch_size=64,
    epochs=5,
    validation_data=(x_val, y_val)
)

Epoch 1/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.6570 - val_loss: 0.6201
Epoch 2/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - loss: 0.6155 - val_loss: 0.6198
Epoch 3/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.6098 - val_loss: 0.6133
Epoch 4/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - loss: 0.6086 - val_loss: 0.6120
Epoch 5/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.6078 - val_loss: 0.6124


In [ ]:
not_watched[x][0]) for x in top_ratings_indices]

print(f"--- Recommendations for User {user_id} ---")# Pick a random user
user_id = df.userId.sample(1).iloc[0]
movies_watched_by_user = df[df.userId == user_id]
movies_not_watched = movie_df[~movie_df["movieId"].isin(movies_watched_by_user.movieId.values)]["movieId"]
movies_not_watched = list(set(movies_not_watched).intersection(set(movie2movie_encoded.keys())))
movies_not_watched = [[movie2movie_encoded.get(x)] for x in movies_not_watched]

user_encoder = user2user_encoded.get(user_id)
user_movie_array = np.hstack(([[user_encoder]] * len(movies_not_watched), movies_not_watched))

# Predict and show results
ratings = model.predict(user_movie_array).flatten()
top_ratings_indices = ratings.argsort()[-10:][::-1]
recommended_movie_ids = [movie_encoded2movie.get(movies_
recommended_movies = movie_df[movie_df["movieId"].isin(recommended_movie_ids)]
for row in recommended_movies.itertuples():
    print(f"{row.title}: {row.genres}")

262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
--- Recommendations for User 274 ---
Boot, Das (Boat, The) (1981): Action|Drama|War
Glory (1989): Drama|War
Miller's Crossing (1990): Crime|Drama|Film-Noir|Thriller
Graduate, The (1967): Comedy|Drama|Romance
Touch of Evil (1958): Crime|Film-Noir|Thriller
Femme Nikita, La (Nikita) (1990): Action|Crime|Romance|Thriller
Bridge on the River Kwai, The (1957): Adventure|Drama|War
Chinatown (1974): Crime|Film-Noir|Mystery|Thriller
Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001): Comedy|Romance
Fog of War: Eleven Lessons from the Life of Robert S. McNamara, The (2003): Documentary|War
